In [1]:
%load_ext autoreload
%autoreload 2

In [56]:
#!/usr/bin/env python
# Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
# SPDX-License-Identifier: Apache-2.0
import sys 
sys.path.append('/home/hbz15/orthogonal-additive-gaussian-processes')  # Adjust path to import oak_model

import numpy as np
import tensorflow as tf
import gpflow
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

from oak.model_utils import oak_model


# ---------------------------------------------------------------------
# 1.  Regression smoke test – **one configuration only**
# ---------------------------------------------------------------------
np.random.seed(44)
tf.random.set_seed(44)

# synthetic 3-D data
N = 500
# X = np.random.normal(0, 1, (N, 3))[:,:1]
X = np.random.normal(0, 1, (N, 4))
y = (
    X[:, 0]
    + X[:, 1]
    + X[:, 2]*X[:, 3]
    + np.random.normal(0, 0.1, N)
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y[:, None], test_size=0.2, random_state=42
)

# --- the one kernel we want:  dim0 (1-D)  +  dims1&2 (2-D) ---------
base_kernels = [
    gpflow.kernels.RBF,                                  # dim 0
    gpflow.kernels.RBF,                                  # dim 0
    lambda **kw: gpflow.kernels.RBF(),                                  # dim 0
]
active_dims = [
    [0],
    [1],
    [2,3]
]
# 3 inputs total → you must give 3 entries
# after (FIX)
# base_kernels = [
    # dim 0 → its own RBF
    # gpflow.kernels.RBF,
    # dim 1 & 2 → the 2-D RBF you want
    # lambda **kw: gpflow.kernels.RBF(lengthscales=[0.7,1.3], variance=2.0, **kw),
# ]

# active_dims = [
    # [0],     # first block: dimension 0
    # [1, 2],  # second block: dimensions 1 & 2
# ]


oak = oak_model(
    num_inducing=50,
    max_interaction_depth=2,
    use_sparsity_prior=True,
    sparse=False,
    base_kernels=base_kernels,
    active_dims=active_dims,
)
oak.fit(
    X_train,
    y_train,
    initialise_inducing_points=True,
    optimise=True,
)

y_pred = oak.predict(X_test, clip=False)
rss = mean_squared_error(y_pred[:, None], y_test)
baseline = mean_squared_error(y_pred[:, None], y_test)

print("\n=== Regression demo (custom 1-D + 2-D blocks) ===")
print(f"RSS = {rss:.4f}   baseline = {baseline:.4f}")


sobol = oak.get_sobol(likelihood_variance=False)
print("\n=== Sobol demo (1-D + 2-D blocks) ===")
print("Sobol indices:", sobol)
print("Sum:", sobol.sum())


indices of binary feature  []
indices of continuous feature  [0, 1, 2, 3]
indices of categorical feature  []
OrthogonalRBFKernel: mu=0.0, var=1.0, D=1
OrthogonalRBFKernel: mu=0.0, var=1.0, D=1
OrthogonalRBFKernel: mu=0.0, var=1.0, D=2
Optimisation took 2.6s

=== Regression demo (custom 1-D + 2-D blocks) ===
RSS = 0.0119   baseline = 0.0119

=== Sobol demo (1-D + 2-D blocks) ===
Sobol indices: [2.89070930e-01 2.55483229e-01 4.55445841e-01 2.18239670e-25
 1.54363329e-24 2.95120334e-24]
Sum: 1.0


In [ ]:
oak.X_scaled.mean(0), oak.X_scaled.std(0)

(array([ 3.55525083e-05,  1.52994740e-03, -2.45514483e-03,  2.01738597e-03]),
 array([0.99773529, 0.99681337, 0.99612857, 1.00136545]))

In [ ]:
oak.m.kernel.kernels[-1]

name,class,transform,prior,trainable,shape,dtype,value
OrthogonalRBFKernel.base_kernel.lengthscales,Parameter,Sigmoid,,True,(),float64,1


In [ ]:
oak.X_scaled[:,-1]

array([-1.46096311,  2.12328801, -1.33489041, -0.48658358, -1.83675421,
        0.89826979,  0.16914489,  0.96179355, -0.18462176,  1.75257659,
       -0.47523422,  0.75894625,  0.56802543, -0.23429887, -0.309709  ,
        1.57197698, -0.72321631,  1.33180133,  1.01191348,  0.54381901,
        1.21097561,  0.72224988, -0.19632011, -0.31247958, -0.77489778,
       -0.58303688,  0.39428451, -0.13066772, -0.28265356,  2.34324518,
       -0.88092225,  0.942832  ,  0.50579474,  0.63864909, -0.64353861,
        0.39520206,  0.12601873, -1.66787856, -0.61058983, -0.60270753,
       -1.06399385, -0.7112195 , -0.00949885, -0.31021558,  0.64328881,
        0.4358117 ,  0.34522627, -0.6735554 , -0.5381023 , -0.37749417,
       -1.28440793,  1.13482657,  0.06964857, -0.51135195,  1.48351888,
       -0.44451647,  0.42589604,  0.55187478,  0.42341557,  0.02586611,
        1.55281091, -0.32179441, -1.15591633,  1.57285907, -0.80298368,
       -0.90092188,  0.936682  ,  0.66303075,  1.43768718, -1.57

In [59]:
oak = oak_model(
    num_inducing=50,
    max_interaction_depth=2,
    use_sparsity_prior=True,
    sparse=False,
    base_kernels=base_kernels,
    active_dims=active_dims,
)
oak.fit(
    X_train,
    y_train,
    initialise_inducing_points=True,
    optimise=False,
)
oak.predict(X_test, clip=False)

indices of binary feature  []
indices of continuous feature  [0, 1, 2, 3]
indices of categorical feature  []
[None, None, None, None] [None, None, None, None]
OrthogonalRBFKernel: mu=0.0, var=1.0, D=1
OrthogonalRBFKernel: mu=0.0, var=1.0, D=1
OrthogonalRBFKernel: mu=0.0, var=1.0, D=2


array([-3.68985644,  0.33870784, -1.96849847,  3.53804686, -1.42235729,
       -0.95676807, -0.306454  ,  0.8618373 , -0.32697513,  2.27728131,
       -1.98318905, -1.62144534,  1.32946757,  1.17461111,  1.96343278,
        0.39344515,  0.2506815 , -2.75022038,  1.30930585, -1.70347376,
       -0.58194179, -1.01506041,  0.79960758,  0.49357782,  2.4255664 ,
        1.20059871, -2.79895846,  1.34695606, -0.06354658, -1.83793452,
       -0.75183785,  0.33491949,  0.50258302, -0.64665355,  0.70834412,
       -1.02253942,  1.58186777, -2.74632457,  1.35952106, -1.27289623,
        0.57770823,  3.12296263, -1.53573088, -0.88488954, -0.69305819,
        1.52563931, -1.25407715,  0.54808977, -2.45816867, -2.90358207,
        0.62149702, -2.87177535,  0.67272754, -1.04971357,  3.42284568,
        1.08216623,  1.44275708, -1.49354328, -0.43119521, -1.60648923,
        1.0584936 , -3.12827927,  0.28628156,  1.91248958,  0.09077629,
       -1.19108164, -2.12728988, -1.26824442, -0.82681511, -0.96

In [48]:
oak.X_scaled[:,0]

array([-8.32071121e-01,  1.21784088e+00, -1.00387886e+00, -4.46352065e-01,
        2.76526703e+00,  2.17042375e+00, -1.10581504e+00,  2.06091756e+00,
       -6.98768180e-01, -6.29703983e-01,  1.70634712e-01,  5.36455045e-01,
       -6.22373736e-01,  1.50572799e+00,  1.01703054e+00, -1.41240341e+00,
        2.86859159e-01,  3.71710633e-01,  1.04527144e+00,  7.25845980e-01,
       -1.23729654e+00,  1.00321137e-01, -2.83727187e-01, -8.11228892e-01,
        1.80059685e+00,  8.50938648e-01,  3.37199521e-01,  1.91409936e-01,
        8.64305629e-02,  2.86388394e-01,  2.10164449e+00,  9.58946765e-01,
        9.31547640e-01, -3.50203870e-01,  5.67016834e-01, -4.04919248e-01,
        9.64411736e-02,  5.65377880e-01,  1.27818950e+00, -1.96652083e-01,
        3.82104465e-01, -3.18804131e-02,  3.95603449e-01, -4.35540476e-01,
       -1.98436714e+00, -3.78555030e-01, -2.25543759e+00, -4.13048721e-01,
        2.02534909e-01,  5.35855128e-01, -1.13243268e+00, -6.08243376e-01,
        5.25077436e-01,  

In [49]:
oak.X_scaled[:-2:]

array([[-0.83207112, -0.17491797, -0.28113821, -1.479903  ],
       [ 1.21784088, -0.17116728, -0.14993879,  2.12861399],
       [-1.00387886, -0.734132  , -0.91414111, -1.38290468],
       ...,
       [ 0.32287949, -1.77582314, -1.41716517, -0.2364417 ],
       [ 0.99519315, -0.70668374,  1.15775811,  0.4369584 ],
       [-0.45500083, -0.88191827,  0.82946376, -2.17212503]])

In [ ]:
X_transformed_2d = oak._transform_x(X_test)[:, -2:]
# calculate correlation between the transformed 2D inputs
correlation_matrix = np.corrcoef(X_transformed_2d.T)
print("Correlation matrix of transformed 2D inputs:")
print(correlation_matrix)

Correlation matrix of transformed 2D inputs:
[[1.         0.02771307]
 [0.02771307 1.        ]]


In [4]:
oak.m.kernel.kernels[-1].slice(oak.X_scaled)

(<tf.Tensor: shape=(400, 2), dtype=float64, numpy=
 array([[-2.86161502e-01, -1.46096311e+00],
        [-1.56700154e-01,  2.12328801e+00],
        [-9.12577574e-01, -1.33489041e+00],
        [-4.44037729e-01, -4.86583583e-01],
        [-1.09032030e+00, -1.83675421e+00],
        [-1.58983321e+00,  8.98269788e-01],
        [ 1.78718539e+00,  1.69144891e-01],
        [-1.31772317e+00,  9.61793546e-01],
        [ 1.12174806e+00, -1.84621757e-01],
        [-9.98261502e-02,  1.75257659e+00],
        [ 3.84368381e-01, -4.75234224e-01],
        [-6.38833631e-01,  7.58946247e-01],
        [ 1.86640823e-01,  5.68025434e-01],
        [ 8.59891914e-01, -2.34298873e-01],
        [-3.98636930e-01, -3.09708998e-01],
        [-1.05508702e+00,  1.57197698e+00],
        [-1.25989550e+00, -7.23216309e-01],
        [ 4.53768767e-01,  1.33180133e+00],
        [-3.84295155e-01,  1.01191348e+00],
        [-4.67911820e-01,  5.43819013e-01],
        [-8.92151683e-01,  1.21097561e+00],
        [-1.69111159e+00,

In [5]:
y_pred = oak.predict(X_test, clip=False)
rss = mean_squared_error(y_pred[:, None], y_test)
baseline = mean_squared_error(y_pred[:, None], y_test)
print("\n=== Regression demo (custom 1-D + 2-D blocks, no optimisation) ===")
print(f"RSS = {rss:.4f}   baseline = {baseline:.4f}")

# plot prediction vs ground truth
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Ground Truth')
plt.ylabel('Predicted')
plt.title('Prediction vs Ground Truth')
plt.grid()
plt.show()

2025-07-08 16:08:43.135613: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.


ValueError: Input contains NaN, infinity or a value too large for dtype('float64').

In [ ]:
oak.X_scaled

In [ ]:
normalised_sobols = oak.get_sobol(likelihood_variance=False)

# can you annotate the Sobol indices normalised_sobols with respect to the blocks in the code
print("\n=== Normalised Sobol indices ===")
print("Normalised Sobol indices:", normalised_sobols)

# what do each element in normalised_sobols mean?
# - The first element corresponds to the Sobol index for the first block (1-D kernel).
# - The second element corresponds to the Sobol index for the second block (2-D kernel).
# - The third element corresponds to the interaction term between the first and second blocks.

In [ ]:
oak.plot()